# 💳 Financial Transaction Fraud Detection System
### scikit-learn (classification + anomaly detection) · SQLite · Streamlit

This notebook builds the **first** project from your resume:

> *Developed a machine learning system to identify potentially fraudulent financial
> transactions using transaction and customer behavior features. Implemented feature
> engineering and classification/anomaly-detection techniques to identify unusual
> transaction patterns. Built an interactive dashboard displaying transaction risk
> scores, suspicious transactions, and key fraud indicators.*

**Dataset:** the real Kaggle "Credit Card Fraud Detection" dataset — 284,807 European
credit card transactions from September 2013, of which only **492 (0.17%) are fraud**.
This extreme imbalance is the single most important fact about this project and shapes
every decision below.

**What you'll build, in order:**
1. Feature engineering on top of the raw transaction data.
2. A **classifier** (Random Forest) trained on labeled past fraud — plus a simple
   Logistic Regression baseline to see the trade-offs.
3. An **anomaly detector** (Isolation Forest) that needs *no* labels at all — useful for
   catching fraud patterns never seen before.
4. Honest evaluation using metrics that actually mean something on imbalanced data
   (accuracy would be a trap here — predicting "not fraud" for everything already
   scores 99.8%).
5. A **Streamlit dashboard** showing risk scores, an adjustable flag-threshold with a
   live precision/recall trade-off, the most suspicious transactions, and which
   features the model actually relies on.


In [7]:
from google.colab import files

uploaded = files.upload()

Saving creditcard.csv to creditcard.csv


In [5]:
!pip install -q scikit-learn joblib streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 41.3 MB/s eta 0:00:00


In [8]:
import pandas as pd

df = pd.read_csv("creditcard.csv")
print("Shape:", df.shape)
print()
print("Class balance:")
print(df["Class"].value_counts())
print()
print("Fraud rate: {:.4f}%".format(100 * df["Class"].mean()))
print()
print("Any missing values?", df.isnull().sum().sum())
df.head()

Shape: (284807, 31)

Class balance:
Class
0    284315
1       492
Name: count, dtype: int64

Fraud rate: 0.1727%

Any missing values? 0


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 4. Feature engineering, training, and evaluation

This is the core of the project. The script below:
- Engineers a few extra features (`hour_of_day`, a log-scaled amount, and a rough
  "how busy was this time period" signal) — see the code comments for an honest note on
  why true *per-customer* behavioral features aren't possible with this specific public
  dataset (no customer ID is included, for privacy).
- Trains a **Logistic Regression baseline** (simple, interpretable, a useful comparison
  point).
- Trains the main **Random Forest classifier**, using `class_weight="balanced"` so it
  doesn't just learn to lazily predict "not fraud" every time.
- Trains an **Isolation Forest anomaly detector** — this one never sees the fraud labels
  at all, so its ability to catch anything is a genuinely different (and complementary)
  signal from the classifier.
- Evaluates all three with **precision, recall, F1, ROC-AUC, and PR-AUC** — and prints a
  confusion matrix so you can see exactly how many frauds were caught vs missed.
- Saves the trained model and writes the held-out test set (never seen during training),
  scored with a 0-100 risk score, into a SQLite database for the dashboard.


In [9]:
%%writefile fraud_model.py
"""
Financial Transaction Fraud Detection System — model training
----------------------------------------------------------------
Dataset: the real Kaggle "Credit Card Fraud Detection" dataset — 284,807
European credit card transactions from September 2013, 492 of them (0.17%)
confirmed fraud. Features V1-V28 are already anonymized via PCA (this is
real customer data, so the bank/Kaggle scrubbed the original fields before
release) — Time and Amount are the only two features in their original form.

This script:
  1. Loads the data and engineers a few extra features from what's available.
  2. Trains a supervised CLASSIFIER (Random Forest) — needs labeled fraud
     examples, learns exactly what past fraud looked like.
  3. Trains an unsupervised ANOMALY DETECTOR (Isolation Forest) — needs no
     labels at all, just flags transactions that look statistically unusual.
     This matters in the real world because new fraud patterns won't match
     anything the classifier was trained on; an anomaly detector can catch
     those "never seen before" cases the classifier would miss.
  4. Evaluates both properly for an imbalanced problem (precision, recall,
     F1, ROC-AUC, and PR-AUC — plain accuracy would be meaningless here,
     since predicting "not fraud" for everything already scores 99.8%).
  5. Saves the trained model + scaler, and writes the held-out test set
     (with risk scores attached) into a SQLite database for the dashboard.
"""

import sqlite3
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    confusion_matrix,
)

CSV_PATH = "creditcard.csv"
DB_PATH = "fraud_transactions.db"
MODEL_PATH = "fraud_model.pkl"
SCALER_PATH = "scaler.pkl"
RANDOM_STATE = 42


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add a few features on top of the raw columns.

    Note: this public dataset has no customer/account ID, so true
    per-customer behavioral features (this resume bullet's "customer
    behavior features") aren't possible here — a real bank system would
    join in account history. We approximate what signal we can from what's
    actually available: time-of-day and how busy the overall transaction
    stream was around each transaction.
    """
    df = df.copy()
    df["hour_of_day"] = (df["Time"] % 86400) // 3600
    df["amount_log"] = np.log1p(df["Amount"])

    # Rolling count of transactions in the same hour-bucket, as a coarse
    # proxy for "unusually busy period" (a real system would do this per
    # account; here it's dataset-wide, which is an honest limitation).
    hour_bucket = (df["Time"] // 3600).astype(int)
    txns_per_hour = hour_bucket.value_counts()
    df["txns_in_hour_bucket"] = hour_bucket.map(txns_per_hour)

    return df


def load_and_prepare():
    df = pd.read_csv(CSV_PATH)
    df = engineer_features(df)

    feature_cols = [c for c in df.columns if c not in ("Class",)]
    X = df[feature_cols]
    y = df["Class"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    scaler = StandardScaler()
    # Only scale Amount/Time-derived columns — the V1-V28 columns are
    # already PCA-transformed (mean ~0, unit-ish variance) by Kaggle.
    scale_cols = ["Time", "Amount", "hour_of_day", "amount_log", "txns_in_hour_bucket"]
    X_train = X_train.copy()
    X_test = X_test.copy()
    X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
    X_test[scale_cols] = scaler.transform(X_test[scale_cols])

    return X_train, X_test, y_train, y_test, feature_cols, scaler


def train_classifier(X_train, y_train):
    # class_weight="balanced" tells the model to pay much more attention to
    # the rare fraud class instead of just learning to always predict "not
    # fraud" and being right 99.8% of the time by default.
    clf = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    clf.fit(X_train, y_train)
    return clf


def train_baseline(X_train, y_train):
    """A simple, interpretable baseline to compare the Random Forest against."""
    base = LogisticRegression(class_weight="balanced", max_iter=2000)
    base.fit(X_train, y_train)
    return base


def train_anomaly_detector(X_train, y_train):
    # Isolation Forest never sees the labels — it only learns what "normal"
    # looks like from the (mostly non-fraud) data, then flags whatever
    # doesn't fit that pattern. contamination is set to the real fraud rate
    # so it flags a realistic proportion of transactions.
    contamination = y_train.mean()
    iso = IsolationForest(
        n_estimators=200, contamination=contamination, random_state=RANDOM_STATE
    )
    iso.fit(X_train)
    return iso


def evaluate(name, y_true, y_pred, y_proba):
    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, target_names=["legit", "fraud"], digits=3))
    print("ROC-AUC:", round(roc_auc_score(y_true, y_proba), 4))
    print("PR-AUC (average precision):", round(average_precision_score(y_true, y_proba), 4))
    print("Confusion matrix [[TN FP] [FN TP]]:")
    print(confusion_matrix(y_true, y_pred))


def main():
    print("Loading and preparing data...")
    X_train, X_test, y_train, y_test, feature_cols, scaler = load_and_prepare()
    print(f"Train: {len(X_train):,} rows ({y_train.sum()} fraud) | "
          f"Test: {len(X_test):,} rows ({y_test.sum()} fraud)")

    print("\nTraining baseline (Logistic Regression)...")
    baseline = train_baseline(X_train, y_train)
    base_proba = baseline.predict_proba(X_test)[:, 1]
    base_pred = (base_proba >= 0.5).astype(int)
    evaluate("Logistic Regression (baseline)", y_test, base_pred, base_proba)

    print("\nTraining classifier (Random Forest)...")
    clf = train_classifier(X_train, y_train)
    clf_proba = clf.predict_proba(X_test)[:, 1]
    clf_pred = (clf_proba >= 0.5).astype(int)
    evaluate("Random Forest (main classifier)", y_test, clf_pred, clf_proba)

    print("\nTraining anomaly detector (Isolation Forest, unsupervised)...")
    iso = train_anomaly_detector(X_train, y_train)
    # IsolationForest scores: lower (more negative) = more anomalous.
    # decision_function gives a continuous score; predict gives -1/1.
    iso_raw_scores = -iso.decision_function(X_test)  # flip sign: higher = more suspicious
    iso_pred = (iso.predict(X_test) == -1).astype(int)
    # Normalize the raw scores to a 0-1 range so they're comparable to a probability.
    iso_proba = (iso_raw_scores - iso_raw_scores.min()) / (iso_raw_scores.max() - iso_raw_scores.min())
    evaluate("Isolation Forest (anomaly detector, no labels used)", y_test, iso_pred, iso_proba)

    print("\nSaving model artifacts...")
    joblib.dump(clf, MODEL_PATH)
    joblib.dump(scaler, SCALER_PATH)
    joblib.dump(feature_cols, "feature_cols.pkl")
    joblib.dump(["Time", "Amount", "hour_of_day", "amount_log", "txns_in_hour_bucket"], "scale_cols.pkl")

    print("Writing held-out test set with risk scores to SQLite...")
    results = X_test.copy()
    # Unscale Time/Amount back to human-readable values for the dashboard
    results[["Time", "Amount", "hour_of_day", "amount_log", "txns_in_hour_bucket"]] = \
        scaler.inverse_transform(results[["Time", "Amount", "hour_of_day", "amount_log", "txns_in_hour_bucket"]])
    results["actual_class"] = y_test.values
    results["risk_score"] = (clf_proba * 100).round(2)
    results["anomaly_flag"] = iso_pred
    results["transaction_id"] = [f"TXN{100000+i}" for i in range(len(results))]
    results = results.reset_index(drop=True)

    conn = sqlite3.connect(DB_PATH)
    results.to_sql("transactions", conn, if_exists="replace", index=False)
    conn.close()
    print(f"Saved {len(results):,} scored transactions -> {DB_PATH}")
    print("\nDone.")


if __name__ == "__main__":
    main()


Overwriting fraud_model.py


In [13]:
!python fraud_model.py

Loading and preparing data...
Train: 227,845 rows (394 fraud) | Test: 56,962 rows (98 fraud)

Training baseline (Logistic Regression)...

=== Logistic Regression (baseline) ===
              precision    recall  f1-score   support

       legit      1.000     0.974     0.987     56864
       fraud      0.058     0.918     0.109        98

    accuracy                          0.974     56962
   macro avg      0.529     0.946     0.548     56962
weighted avg      0.998     0.974     0.985     56962

ROC-AUC: 0.9725
PR-AUC (average precision): 0.7221
Confusion matrix [[TN FP] [FN TP]]:
[[55394  1470]
 [    8    90]]

Training classifier (Random Forest)...

=== Random Forest (main classifier) ===
              precision    recall  f1-score   support

       legit      1.000     1.000     1.000     56864
       fraud      0.842     0.816     0.829        98

    accuracy                          0.999     56962
   macro avg      0.921     0.908     0.914     56962
weighted avg      0.999  

**How to read those results:** for the Random Forest, look at the `fraud` row's
`precision` and `recall` — those are the numbers that matter here, not `accuracy`. Compare
them against the Logistic Regression baseline above it to see the classic imbalanced-data
trade-off: the baseline catches more fraud (higher recall) but drowns you in false alarms
(much lower precision). The Isolation Forest numbers show what's achievable with **zero**
labeled examples — meaningfully better than random guessing, but well below the supervised
classifier, which is exactly the trade-off you'd expect between the two approaches.

## 5. Peek at what got saved

Quick sanity check before moving to the dashboard.


In [14]:
import sqlite3

conn = sqlite3.connect("fraud_transactions.db")
result = pd.read_sql("SELECT * FROM transactions ORDER BY risk_score DESC LIMIT 10", conn)
conn.close()
result[["transaction_id", "Amount", "hour_of_day", "risk_score", "anomaly_flag", "actual_class"]]

,transaction_id,Amount,hour_of_day,risk_score,anomaly_flag,actual_class
0,TXN137511,118.30,11.0,99.98,1,1
1,TXN100840,0.01,15.0,99.97,0,1
2,TXN101146,1.00,18.0,99.97,1,1
3,TXN107299,1.63,2.0,99.97,1,1
4,TXN131804,1.00,18.0,99.97,1,1
5,TXN140860,1.00,18.0,99.97,0,1
6,TXN143032,99.99,7.0,99.97,1,1
7,TXN146841,99.99,7.0,99.97,1,1
8,TXN151243,316.06,2.0,99.97,1,1
9,TXN152728,1.00,5.0,99.97,1,1


## 6. The Streamlit dashboard

Two tabs:
- **📊 Overview** — dataset stats, a risk-score distribution chart (fraud vs. legit), an
  interactive slider to explore the precision/recall trade-off at different flagging
  thresholds live, and a feature-importance chart showing what the model actually relies on.
- **🔍 Investigate Transactions** — the most suspicious transactions sorted by risk score,
  a table of where the classifier and the anomaly detector *disagree* (a genuinely useful
  real-world signal — disagreement often means "worth a second look"), and a transaction
  ID lookup box.


In [15]:
%%writefile app.py
"""
Streamlit Dashboard — Financial Transaction Fraud Detection System
---------------------------------------------------------------------
Run locally:   streamlit run app.py
From Colab:    see the notebook's dashboard section (uses pyngrok)

Reads the already-trained model + the held-out, risk-scored test set that
fraud_model.py produces. Nothing here re-trains anything — this is a fast,
read-only dashboard on top of work already done, exactly like a real fraud
analyst's tool would be.
"""

import sqlite3
import joblib
import numpy as np
import pandas as pd
import streamlit as st

DB_PATH = "fraud_transactions.db"
MODEL_PATH = "fraud_model.pkl"
FEATURE_COLS_PATH = "feature_cols.pkl"

st.set_page_config(page_title="Fraud Detection Dashboard", layout="wide")
st.title("💳 Financial Transaction Fraud Detection System")
st.caption("Random Forest classifier + Isolation Forest anomaly detector · scikit-learn · SQLite · Streamlit")


@st.cache_data
def load_data():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("SELECT * FROM transactions", conn)
    conn.close()
    return df


@st.cache_resource
def load_model():
    clf = joblib.load(MODEL_PATH)
    feature_cols = joblib.load(FEATURE_COLS_PATH)
    return clf, feature_cols


try:
    df = load_data()
    clf, feature_cols = load_model()
except FileNotFoundError:
    st.error(
        "Model files not found. Run `python fraud_model.py` first (or the "
        "matching notebook cells) to train the model and create the database."
    )
    st.stop()

tab_overview, tab_investigate = st.tabs(["📊 Overview", "🔍 Investigate Transactions"])

# ---------------------------------------------------------------------------
# Overview tab
# ---------------------------------------------------------------------------
with tab_overview:
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Transactions (held-out test set)", f"{len(df):,}")
    col2.metric("Actual frauds", int(df["actual_class"].sum()))
    col3.metric("Fraud rate", f"{100 * df['actual_class'].mean():.3f}%")
    col4.metric("Avg. transaction amount", f"${df['Amount'].mean():.2f}")

    st.subheader("Risk score distribution: legit vs. actual fraud")
    st.caption(
        "A good model should push fraud (orange) toward high risk scores and "
        "legit transactions (blue) toward low ones. Overlap is where mistakes happen."
    )
    hist_data = pd.DataFrame({
        "risk_score": df["risk_score"],
        "label": df["actual_class"].map({0: "Legit", 1: "Fraud"}),
    })
    st.bar_chart(
        hist_data.assign(bucket=(hist_data["risk_score"] // 10 * 10))
        .groupby(["bucket", "label"]).size().unstack(fill_value=0)
    )

    st.subheader("Pick a risk-score threshold and see the trade-off live")
    threshold = st.slider(
        "Flag any transaction with risk score ≥ this as fraud",
        min_value=0.0, max_value=100.0, value=50.0, step=1.0,
    )
    flagged = df["risk_score"] >= threshold
    actual = df["actual_class"] == 1

    tp = int((flagged & actual).sum())
    fp = int((flagged & ~actual).sum())
    fn = int((~flagged & actual).sum())
    tn = int((~flagged & ~actual).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Flagged as fraud", int(flagged.sum()))
    m2.metric("Precision", f"{precision:.1%}", help="Of what we flagged, how much was really fraud")
    m3.metric("Recall", f"{recall:.1%}", help="Of all real fraud, how much did we catch")
    m4.metric("Missed frauds", fn, help="Real fraud we did NOT flag at this threshold")

    st.caption(
        "Lower the threshold to catch more fraud (higher recall) at the cost of more "
        "false alarms (lower precision) — this trade-off is the central decision a "
        "bank has to make, and it's a business choice, not a purely technical one."
    )

    st.subheader("What the model actually looks at (feature importance)")
    importances = pd.Series(clf.feature_importances_, index=feature_cols)
    top_features = importances.sort_values(ascending=False).head(12)
    st.bar_chart(top_features)
    st.caption(
        "V1-V28 are anonymized (PCA-transformed) features from the original bank "
        "data — we can see *how much* each one matters, but not what it originally "
        "represented, since the bank scrubbed that for privacy before releasing this data."
    )

# ---------------------------------------------------------------------------
# Investigate tab
# ---------------------------------------------------------------------------
with tab_investigate:
    st.subheader("Most suspicious transactions")
    n_show = st.slider("How many to show", 10, 200, 30)
    suspicious = df.sort_values("risk_score", ascending=False).head(n_show)
    display_cols = ["transaction_id", "Amount", "hour_of_day", "risk_score",
                     "anomaly_flag", "actual_class"]
    st.dataframe(
        suspicious[display_cols].rename(columns={
            "actual_class": "was_actually_fraud",
            "anomaly_flag": "flagged_as_anomaly",
        }),
        use_container_width=True, hide_index=True,
    )

    st.subheader("Where the classifier and the anomaly detector disagree")
    st.caption(
        "The Random Forest learned from past labeled fraud; the Isolation Forest "
        "never saw any labels and just flags statistically unusual transactions. "
        "When they disagree, it's often worth a second look — the anomaly detector "
        "might be catching a new pattern the classifier has never seen before."
    )
    disagree = df[(df["risk_score"] >= 50) != (df["anomaly_flag"] == 1)]
    st.dataframe(
        disagree[display_cols].rename(columns={
            "actual_class": "was_actually_fraud",
            "anomaly_flag": "flagged_as_anomaly",
        }).head(50),
        use_container_width=True, hide_index=True,
    )

    st.subheader("Look up a specific transaction")
    txn_id = st.text_input("Transaction ID (e.g. TXN100005)")
    if txn_id:
        row = df[df["transaction_id"] == txn_id]
        if row.empty:
            st.warning("No transaction with that ID in this dataset.")
        else:
            st.write(row[display_cols].rename(columns={
                "actual_class": "was_actually_fraud",
                "anomaly_flag": "flagged_as_anomaly",
            }))


Writing app.py


In [16]:
import ast
ast.parse(open("app.py").read())
print("app.py looks syntactically correct ✅")

app.py looks syntactically correct ✅


## 7. Launch the dashboard (from Colab)

Same approach as your other project: `pyngrok` tunnels a public URL to the Streamlit app
running inside Colab. Free account at https://dashboard.ngrok.com/signup, authtoken from
https://dashboard.ngrok.com/get-started/your-authtoken (same account/token works for both
projects — you don't need a second one).


In [21]:
from getpass import getpass
from pyngrok import ngrok

NGROK_AUTHTOKEN = getpass("Paste your ngrok authtoken: ")
ngrok.set_auth_token(NGROK_AUTHTOKEN)

for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_url = ngrok.connect(8501)
print("Your dashboard is live at:", public_url)

Paste your ngrok authtoken: ··········
Your dashboard is live at: NgrokTunnel: "https://lying-yarn-pointed.ngrok-free.dev" -> "http://localhost:8501"


In [22]:
import subprocess, os

proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    env=os.environ.copy(),
)
print("Streamlit is starting... wait about 10-15 seconds, then click the ngrok link above.")

Streamlit is starting... wait about 10-15 seconds, then click the ngrok link above.
